# v4 → Carlsen: player-specific fine-tuningTakes the trained **v4 Chessformer** (`chess_gpt_v4.pt`) and keeps training it onMagnus Carlsen's games, so the policy shifts from *"the move an average 2200-2600human plays"* toward *"the move Carlsen plays"*. The model already knows how toplay chess; this only re-aims it.Runs on Colab. Set **Runtime → Change runtime type → GPU** first, then upload three files:| File | What it is ||---|---|| `chess_gpt_v4.pt` | your trained v4 weights || `chess_move_vocab_v4.json` | the vocab + architecture config written alongside them || `Carlsen.pgn` | ~4,300 Carlsen games |Output: `chess_gpt_v4_carlsen.pt` + `chess_move_vocab_v4_carlsen.json`, downloaded atthe end. They drop straight into `app_transformer.py` via the `CHESS_MODEL_PATH` /`CHESS_VOCAB_PATH` environment variables — **v4's own artifacts are never overwritten.**---### Three decisions in here that are not defaults**1. Only Carlsen's own moves are trained on and scored.** Each of his games holds ~40of his moves and ~40 of his opponent's. Training on all of them fits *"a strong GM in aCarlsen game"*, not Carlsen. Set `TRAIN_BOTH_SIDES = True` to disable.**2. The baseline is measured in this same notebook, on the same held-out games.**v4's Lichess Elite Top-1 (~0.43) and accuracy on Carlsen's moves are **differentquantities** — predicting one player is not the same task as predicting the averagehuman, and elite play is more predictable, so the Carlsen number starts higher. The onlyhonest read is *un-fine-tuned v4* vs *fine-tuned v4* on identical positions, whichSection 8 measures before training and Section 10 measures after.**3. Ratings are clamped into the pretrained [2200, 2600] band, not rescaled.**See Section 4.

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("chess") is None:
    print("installing python-chess ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess"], check=True)

import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import chess
import chess.pgn

try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are cosmetic; never block on them
    class tqdm:
        def __init__(self, iterable=None, **kw):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable or [])
        def update(self, n=1):
            pass
        def close(self):
            pass

# find_spec("google.colab") raises rather than returning None when there is no
# top-level `google` package at all, which is the case off Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}")
else:
    print("  [!] No GPU. Runtime -> Change runtime type -> T4 GPU, or expect ~75 min/epoch.")

## 1. Configuration`LR` is the one number worth thinking about. Pretraining ran at `3e-4` over ~1.2Mpositions; this run sees ~170k. At the pretraining rate the model forgets most of whatthose 1.2M positions bought and ends up worse at both tasks, so fine-tuning runs ~6xlower. If the train/test gap in Section 10 is large, halve `LR` or cut `N_EPOCHS` —do not train longer.

In [ ]:
# ----- Inputs (upload these) --------------------------------------------
PGN_PATH   = "Carlsen.pgn"
BASE_CKPT  = "chess_gpt_v4.pt"
BASE_VOCAB = "chess_move_vocab_v4.json"

# ----- Outputs (never the v4 filenames) ---------------------------------
OUT_CKPT   = "chess_gpt_v4_carlsen.pt"
OUT_VOCAB  = "chess_move_vocab_v4_carlsen.json"

# ----- What counts as a training example --------------------------------
TRAIN_BOTH_SIDES = False   # False = train/score only on positions where Carlsen moves
MIN_MOVES, MAX_MOVES = 8, 200      # same game-length filter as pretraining
CHRONOLOGICAL_SPLIT = False        # True = hold out his most recent 20% instead of a random 20%

# Header spellings that are actually Magnus. "Carlsen,H" (his father Henrik) appears
# once in this file and must NOT match -- a substring test on "carlsen" would have
# silently trained on the wrong player's moves.
CARLSEN_NAMES = {"carlsen,m", "carlsen,magnus", "magnuscarlsen"}

# ----- Fine-tuning ------------------------------------------------------
LR                  = 5e-5     # ~6x below pretraining's 3e-4
WEIGHT_DECAY        = 0.01
N_EPOCHS            = 8
EARLY_STOP_PATIENCE = 2
GAMES_PER_BATCH     = 16       # batch is in GAMES; ~40 Carlsen positions each
GRAD_CLIP           = 1.0
NUM_WORKERS         = 2        # 2 is fine on Colab; use 0 on Windows/Jupyter

# ----- Evaluation -------------------------------------------------------
EVAL_MAX_GAMES       = 400     # held-out games in the Top-k report
EVAL_GAMES_PER_EPOCH = 150     # per-epoch Top-1 probe; 0 to skip

_scope = "all moves" if TRAIN_BOTH_SIDES else "Carlsen's moves only"
print(f"scope: {_scope}")
print(f"lr {LR:.1e}, {N_EPOCHS} epochs, {GAMES_PER_BATCH} games/batch")

## 2. Upload the input filesOn Colab this opens a file picker if anything is missing. Alternatively mount Drive and`os.chdir()` into the folder that already holds them.

In [ ]:
required = [PGN_PATH, BASE_CKPT, BASE_VOCAB]
missing = [p for p in required if not os.path.exists(p)]

if missing and IN_COLAB:
    print("Missing:", ", ".join(missing))
    print("Pick them in the dialog below (you can select all at once).")
    from google.colab import files
    files.upload()
    missing = [p for p in required if not os.path.exists(p)]

if missing:
    raise SystemExit(f"Still missing: {missing}. Upload them, or os.chdir() to their folder.")

for p in required:
    print(f"  {p:<28} {os.path.getsize(p) / 1e6:>7.2f} MB")

## 3. Vocabulary and architecture — read from the artifact, never regeneratedThe action space and the model shape both come out of `chess_move_vocab_v4.json`.Regenerating the vocab here would risk a different ordering, and a checkpoint is onlydecodable with the vocab it was saved beside. Reading the architecture from the samefile means this notebook still works if you retrain v4 with different `USE_*` flags.

In [ ]:
with open(BASE_VOCAB, "r") as f:
    vocab_data = json.load(f)

if vocab_data.get("arch") != "chessformer-v4":
    raise SystemExit(f"{BASE_VOCAB} says arch={vocab_data.get('arch')!r}; this notebook "
                     f"only fine-tunes chessformer-v4 checkpoints.")

itos = vocab_data["itos"]
stoi = {tok: i for i, tok in enumerate(itos)}
VOCAB_SIZE = len(itos)

D_MODEL   = vocab_data["d_model"]
N_HEAD    = vocab_data["n_head"]
N_LAYER   = vocab_data["n_layer"]
USE_GAB   = vocab_data["use_gab"]
USE_HISTORY = vocab_data["use_history"]
K_HISTORY = vocab_data["k_history"]
USE_SQUARE_AUX = vocab_data["use_square_aux"]
STATE_DIM = vocab_data["state_dim"]
META_DIM  = vocab_data["meta_dim"]
AUX_DIM   = vocab_data["aux_dim"]
MIN_ELO, MAX_ELO = vocab_data["min_elo"], vocab_data["max_elo"]
HAS_CLOCK_DATA = vocab_data["has_clock_data"]

DROPOUT   = 0.1                    # unchanged from pretraining, on purpose
N_SPECIAL = 2                      # <PAD>, <BOS> -- kept for index parity with v1-v3
NO_SQUARE = 64                     # sentinel for "no move here yet" in history
N_PIECE_IDS = 13
ELO_SPAN = max(1, MAX_ELO - MIN_ELO)
_TC_LOG_MAX = math.log1p(10800.0)

# ----- move-space decomposition (notebook v4 section 4) -----------------
PROMOTION_PIECES = [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]
PROMO_INDEX = {pc: i for i, pc in enumerate(PROMOTION_PIECES)}
N_PROMO = len(PROMOTION_PIECES)

MOVE_FROM  = np.zeros(VOCAB_SIZE, dtype=np.int64)
MOVE_TO    = np.zeros(VOCAB_SIZE, dtype=np.int64)
MOVE_PROMO = np.full(VOCAB_SIZE, -1, dtype=np.int64)

for _i in range(N_SPECIAL, VOCAB_SIZE):
    _mv = chess.Move.from_uci(itos[_i])
    MOVE_FROM[_i] = _mv.from_square
    MOVE_TO[_i] = _mv.to_square
    if _mv.promotion is not None:
        MOVE_PROMO[_i] = PROMO_INDEX[_mv.promotion]

MOVE_FT       = MOVE_FROM * 64 + MOVE_TO
MOVE_PROMO_FT = np.where(MOVE_PROMO >= 0, MOVE_TO * N_PROMO + np.maximum(MOVE_PROMO, 0), 0)
IS_PROMO      = MOVE_PROMO >= 0

# piece id: 0 = empty, 1-6 = white P N B R Q K, 7-12 = black P N B R Q K
PIECE_ID = {}
for _ci, _color in enumerate((chess.WHITE, chess.BLACK)):
    for _pi, _pt in enumerate((chess.PAWN, chess.KNIGHT, chess.BISHOP,
                               chess.ROOK, chess.QUEEN, chess.KING)):
        PIECE_ID[(_color, _pt)] = 1 + _ci * 6 + _pi

print(f"vocab {VOCAB_SIZE} tokens | d_model={D_MODEL} n_layer={N_LAYER} n_head={N_HEAD}")
print(f"gab={USE_GAB} history={USE_HISTORY}(K={K_HISTORY}) aux={USE_SQUARE_AUX} "
      f"elo band [{MIN_ELO}, {MAX_ELO}]")

## 4. Carlsen.pgn → recordsTwo things about this data that change what gets fed in:**Ratings are clamped into `[MIN_ELO, MAX_ELO]`, not rescaled.** Carlsen sits at 2800+and most of his opponents above 2600, so both saturate at 1.0. Rescaling the band to fitthis dataset would change what the meta vector *means* relative to the pretrained`skill_proj` weights — the exact thing fine-tuning is meant to preserve. Consequence toexpect: the ELO input carries almost no signal on this data, so the rating slider in theweb app will do much less after fine-tuning. ~1% of games leave a rating blank (mostlyCarlsen's own in 2001-2003, when he was ~2000); the fallback is the **opponent's** rating,not his peak — tournament pairings put players of similar strength together.**There is no `TimeControl` header and no `%clk`,** so every record is stamped with180+0 and `has_clk=0`. That is deliberately the same value `app_transformer.py` sends atinference. The model has no representation of "classical OTB" to be told about; whatmatters is that training and serving agree.

In [ ]:
TC_BASE, TC_INC = 180.0, 0.0      # matches app_transformer.py's serving default


def _norm_name(s):
    return (s or "").replace(" ", "").lower()


def _parse_elo(s):
    s = (s or "").strip()
    return int(s) if s.isdigit() else None


def _clamp_elo(e):
    return max(MIN_ELO, min(MAX_ELO, e))


def game_to_record(game):
    """One Carlsen game -> the record shape extract_game_positions() expects."""
    if game.errors:
        return None

    w_is = _norm_name(game.headers.get("White")) in CARLSEN_NAMES
    b_is = _norm_name(game.headers.get("Black")) in CARLSEN_NAMES
    if w_is == b_is:                 # neither side is Magnus (or the headers are broken)
        return None

    w_raw = _parse_elo(game.headers.get("WhiteElo"))
    b_raw = _parse_elo(game.headers.get("BlackElo"))
    if w_raw is None and b_raw is None:
        w_raw = b_raw = MAX_ELO
    elif w_raw is None:
        w_raw = b_raw
    elif b_raw is None:
        b_raw = w_raw

    ucis = []
    for node in game.mainline():
        if node.move is None:
            return None
        ucis.append(node.move.uci())

    if not (MIN_MOVES <= len(ucis) <= MAX_MOVES):
        return None
    if any(u not in stoi for u in ucis):        # null / variant moves
        return None

    date = game.headers.get("Date", "")
    year = int(date[:4]) if date[:4].isdigit() else 0

    return {"uci": ucis, "w": _clamp_elo(w_raw), "b": _clamp_elo(b_raw),
            "clk": [None] * len(ucis), "base": TC_BASE, "inc": TC_INC,
            "carlsen_white": w_is, "year": year, "w_raw": w_raw, "b_raw": b_raw}


def carlsen_mask(rec):
    """(T,) bool -- True at the positions where it is Carlsen's turn to move."""
    T = len(rec["uci"])
    white_to_move = (np.arange(T) % 2 == 0)
    return white_to_move if rec["carlsen_white"] else ~white_to_move


records, skipped = [], 0
t0 = time.time()
with open(PGN_PATH, encoding="utf-8", errors="replace") as f:
    pbar = tqdm(desc="Reading Carlsen.pgn", unit="game")
    while True:
        game = chess.pgn.read_game(f)
        if game is None:
            break
        pbar.update(1)
        rec = game_to_record(game)
        if rec is None:
            skipped += 1
        else:
            records.append(rec)
    pbar.close()

n_white = sum(1 for r in records if r["carlsen_white"])
n_pos = sum(len(r["uci"]) for r in records)
n_carl = int(sum(carlsen_mask(r).sum() for r in records))
years = [r["year"] for r in records if r["year"]]
his_elo = [r["w_raw"] if r["carlsen_white"] else r["b_raw"] for r in records]

print(f"\nGames kept: {len(records)}  (skipped {skipped})  [{time.time() - t0:.0f}s]")
print(f"  Carlsen as White: {n_white}   as Black: {len(records) - n_white}")
print(f"  Positions: {n_pos:,} total, {n_carl:,} where Carlsen is to move")
print(f"  Years {min(years)}-{max(years)}")
print(f"  His rating in the headers: {min(his_elo)}-{max(his_elo)}  "
      f"-> clamped into [{MIN_ELO}, {MAX_ELO}]")

In [ ]:
# Game-level split, never position-level: positions from one game are near-duplicates
# of each other, so splitting inside a game leaks the answer and inflates every number.
if CHRONOLOGICAL_SPLIT:
    records.sort(key=lambda r: r["year"])
    cut = int(0.8 * len(records))
    train_records, test_records = records[:cut], records[cut:]
    print(f"Chronological split: test set starts at {test_records[0]['year']}")
else:
    perm = np.random.RandomState(SEED).permutation(len(records))
    n_test = int(math.ceil(0.2 * len(records)))
    test_records  = [records[i] for i in perm[:n_test]]
    train_records = [records[i] for i in perm[n_test:]]

tr_pos = int(sum(carlsen_mask(r).sum() for r in train_records))
te_pos = int(sum(carlsen_mask(r).sum() for r in test_records))
print(f"Train games: {len(train_records)}  ({tr_pos:,} Carlsen positions)")
print(f"Test games:  {len(test_records)}  ({te_pos:,} Carlsen positions)")

## 5. Feature extractionIdentical to v4's Section 5 — same layout, same causality rules. Features at index `t`describe the position **before** move `t`, and `target[t]` is move `t`. Nothing aboutthis is Carlsen-specific; only the row mask in Section 6 is.

In [ ]:
def build_meta_features(rec, T):
    """(T, META_DIM). Same semantics as v3/v4, including the causal clock lag."""
    meta = np.zeros((T, META_DIM), dtype=np.float32)
    e_w = (rec["w"] - MIN_ELO) / ELO_SPAN
    e_b = (rec["b"] - MIN_ELO) / ELO_SPAN
    base, clks = rec["base"], rec["clk"]
    has_clk = 1.0 if (base and any(c is not None for c in clks)) else 0.0
    tc_base = min(1.0, math.log1p(base) / _TC_LOG_MAX) if base else 0.0
    tc_inc = min(1.0, rec.get("inc", 0.0) / 60.0)

    for t in range(T):
        white_to_move = (t % 2 == 0)
        meta[t, 0] = e_w if white_to_move else e_b
        meta[t, 1] = e_b if white_to_move else e_w
        if has_clk:
            c_stm = clks[t - 2] if t >= 2 else base
            c_opp = clks[t - 1] if t >= 1 else base
            if c_stm is not None:
                meta[t, 2] = min(1.0, c_stm / base)
            if c_opp is not None:
                meta[t, 3] = min(1.0, c_opp / base)
        meta[t, 4] = has_clk
        meta[t, 5] = tc_base
        meta[t, 6] = tc_inc
    return meta


def _square_aux(board, out):
    """(64, AUX_DIM) binary per-square spatial features."""
    out[:] = 0.0
    for j, color in enumerate((chess.WHITE, chess.BLACK)):
        for sq in chess.scan_forward(board.occupied_co[color]):
            for tgt in chess.scan_forward(board.attacks_mask(sq)):
                out[tgt, j] = 1.0
    for mv in board.legal_moves:
        out[mv.from_square, 2] = 1.0
        out[mv.to_square, 3] = 1.0
    return out


def extract_game_positions(rec, with_legal=False):
    """Replay the game once, emitting one training example per position."""
    ucis = rec["uci"]
    T = len(ucis)

    piece_ids = np.zeros((T, 64), dtype=np.int64)
    state     = np.zeros((T, STATE_DIM), dtype=np.float32)
    target    = np.zeros(T, dtype=np.int64)
    hist_from = np.full((T, K_HISTORY), NO_SQUARE, dtype=np.int64)
    hist_to   = np.full((T, K_HISTORY), NO_SQUARE, dtype=np.int64)
    aux       = np.zeros((T, 64, AUX_DIM), dtype=np.float32) if USE_SQUARE_AUX else None
    legal     = [] if with_legal else None

    board = chess.Board()
    for t, uci in enumerate(ucis):
        for sq, piece in board.piece_map().items():
            piece_ids[t, sq] = PIECE_ID[(piece.color, piece.piece_type)]

        state[t, 0] = 1.0 if board.turn == chess.WHITE else 0.0
        state[t, 1] = float(board.has_kingside_castling_rights(chess.WHITE))
        state[t, 2] = float(board.has_queenside_castling_rights(chess.WHITE))
        state[t, 3] = float(board.has_kingside_castling_rights(chess.BLACK))
        state[t, 4] = float(board.has_queenside_castling_rights(chess.BLACK))
        if board.ep_square is not None:
            state[t, 5 + chess.square_file(board.ep_square)] = 1.0

        if USE_HISTORY:
            for k in range(K_HISTORY):
                j = t - 1 - k
                if j < 0:
                    break
                prev = board.move_stack[j]
                hist_from[t, k] = prev.from_square
                hist_to[t, k] = prev.to_square

        if USE_SQUARE_AUX:
            _square_aux(board, aux[t])
        if with_legal:
            legal.append([stoi[m.uci()] for m in board.legal_moves])

        target[t] = stoi[uci]
        board.push(chess.Move.from_uci(uci))

    out = {"piece_ids": piece_ids, "state": state, "meta": build_meta_features(rec, T),
           "hist_from": hist_from, "hist_to": hist_to, "target": target}
    if USE_SQUARE_AUX:
        out["aux"] = aux
    return (out, legal) if with_legal else out


_t0 = time.perf_counter()
_ = extract_game_positions(train_records[0])
print(f"Extraction cost: "
      f"{(time.perf_counter() - _t0) / len(train_records[0]['uci']) * 1e6:.0f} us/position")

### Sanity checksThe failure mode worth guarding against is an off-by-one that leaks move `t` into thefeatures for position `t` — it produces a large, meaningless accuracy rather than acrash. These assertions catch it, plus the Carlsen mask picking the wrong colour.

In [ ]:
_rec = train_records[0]
_d = extract_game_positions(_rec)
_T = len(_rec["uci"])

# Position 0 is the untouched start.
assert (_d["piece_ids"][0] > 0).sum() == 32
assert _d["piece_ids"][0][chess.E1] == PIECE_ID[(chess.WHITE, chess.KING)]
assert _d["state"][0][0] == 1.0                      # White to move
assert (_d["hist_from"][0] == NO_SQUARE).all()       # no history at ply 0

# Features at t must be the board after exactly t moves -- not t+1.
_b = chess.Board()
for _t, _u in enumerate(_rec["uci"]):
    _ref = np.zeros(64, dtype=np.int64)
    for _sq, _pc in _b.piece_map().items():
        _ref[_sq] = PIECE_ID[(_pc.color, _pc.piece_type)]
    assert np.array_equal(_ref, _d["piece_ids"][_t]), f"board misaligned at t={_t}"
    assert _d["state"][_t][0] == (1.0 if _b.turn == chess.WHITE else 0.0)
    assert itos[_d["target"][_t]] == _u, f"target misaligned at t={_t}"
    _b.push(chess.Move.from_uci(_u))

# History token k at position t is the move played k+1 plies ago.
for _t in range(1, min(_T, 12)):
    for _k in range(min(_t, K_HISTORY)):
        _prev = chess.Move.from_uci(_rec["uci"][_t - 1 - _k])
        assert _d["hist_from"][_t, _k] == _prev.from_square
        assert _d["hist_to"][_t, _k] == _prev.to_square

# The mask selects exactly the positions where Carlsen is on move.
_m = carlsen_mask(_rec)
_his_color = chess.WHITE if _rec["carlsen_white"] else chess.BLACK
_b = chess.Board()
for _t, _u in enumerate(_rec["uci"]):
    assert _m[_t] == (_b.turn == _his_color), f"carlsen_mask wrong at t={_t}"
    _b.push(chess.Move.from_uci(_u))

# meta's elo columns are (side-to-move, opponent), not (white, black).
assert np.isclose(_d["meta"][0, 0], (_rec["w"] - MIN_ELO) / ELO_SPAN)
assert np.isclose(_d["meta"][1, 0], (_rec["b"] - MIN_ELO) / ELO_SPAN)

print(f"OK  alignment, history causality, mask and meta verified over {_T} plies")
print(f"    Carlsen played {'White' if _rec['carlsen_white'] else 'Black'} here; "
      f"{int(_m.sum())}/{_T} positions are his")

## 6. Dataset — one item per game, rows filtered to Carlsen's movesBatch size is in **games**; `collate_positions` concatenates their positions into oneflat batch (v4 has no padding, so nothing needs aligning). The row filter is the wholeof what makes this a Carlsen model: everything upstream is version-agnostic.

In [ ]:
class CarlsenPositions(Dataset):
    def __init__(self, records, carlsen_only=True):
        self.records = [r for r in records if len(r["uci"]) >= 1]
        self.carlsen_only = carlsen_only

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        data = extract_game_positions(rec)
        if self.carlsen_only:
            m = carlsen_mask(rec)
            data = {k: v[m] for k, v in data.items()}
        return data


def collate_positions(batch):
    """Concatenate every game's positions into one flat batch. No padding required."""
    out = {}
    for key in batch[0]:
        out[key] = torch.from_numpy(np.concatenate([b[key] for b in batch], axis=0))
    return out


CARLSEN_ONLY = not TRAIN_BOTH_SIDES

train_loader = DataLoader(CarlsenPositions(train_records, CARLSEN_ONLY),
                          batch_size=GAMES_PER_BATCH, shuffle=True,
                          collate_fn=collate_positions, num_workers=NUM_WORKERS)
val_loader   = DataLoader(CarlsenPositions(test_records, CARLSEN_ONLY),
                          batch_size=GAMES_PER_BATCH, shuffle=False,
                          collate_fn=collate_positions, num_workers=NUM_WORKERS)

_b = next(iter(train_loader))
print(f"Train games: {len(train_loader.dataset)}  |  Val games: {len(val_loader.dataset)}")
print(f"Positions in one batch: {_b['target'].shape[0]}")
print(f"Steps per epoch: {len(train_loader)}")

## 7. The modelCopied verbatim from v4's Sections 7-8. It must stay byte-identical in behaviour or thecheckpoint will not load — `strict=True` in the next cell turns any mismatch into acrash instead of a randomly-initialised layer quietly predicting nonsense.

In [ ]:
def build_gab_index(n_square, n_global):
    """(N, N) bucket index. Square pairs -> (df, dr) bucket; anything else -> 3 extra buckets."""
    N = n_square + n_global
    idx = np.zeros((N, N), dtype=np.int64)
    n_geo = 15 * 15
    for i in range(N):
        for j in range(N):
            if i < n_square and j < n_square:
                df = chess.square_file(j) - chess.square_file(i)
                dr = chess.square_rank(j) - chess.square_rank(i)
                idx[i, j] = (df + 7) * 15 + (dr + 7)
            elif i < n_square:
                idx[i, j] = n_geo          # square -> global
            elif j < n_square:
                idx[i, j] = n_geo + 1      # global -> square
            else:
                idx[i, j] = n_geo + 2      # global -> global
    return idx, n_geo + 3


class GeometricSelfAttention(nn.Module):
    """Bidirectional self-attention over board tokens, with a learned per-head bias
    on the geometric relationship between every pair of squares."""

    def __init__(self, d_model, n_head, n_buckets, dropout=DROPOUT, use_gab=True):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.use_gab = use_gab

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.rel_bias = nn.Parameter(torch.zeros(n_head, n_buckets))

    def forward(self, x, gab_index):
        B, N, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.view(B, N, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, N, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, N, self.n_head, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if self.use_gab:
            att = att + self.rel_bias[:, gab_index].unsqueeze(0)

        att = self.dropout(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, N, C)
        return self.proj(out)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_head, n_buckets, dropout=DROPOUT, use_gab=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = GeometricSelfAttention(d_model, n_head, n_buckets, dropout, use_gab)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, gab_index):
        x = x + self.attn(self.ln1(x), gab_index)
        x = x + self.mlp(self.ln2(x))
        return x


class Chessformer(nn.Module):
    """Encoder over 64 square tokens + global tokens, with a bilinear from/to policy head."""

    def __init__(self, d_model=D_MODEL, n_head=N_HEAD, n_layer=N_LAYER, dropout=DROPOUT,
                 use_gab=USE_GAB, use_history=USE_HISTORY, k_history=K_HISTORY,
                 use_aux=USE_SQUARE_AUX):
        super().__init__()
        self.use_history = use_history
        self.use_aux = use_aux
        self.k_history = k_history if use_history else 0

        n_global = 2 + self.k_history
        gab_idx, n_buckets = build_gab_index(64, n_global)
        self.register_buffer("gab_index", torch.from_numpy(gab_idx))
        self.n_tokens = 64 + n_global

        self.piece_emb  = nn.Embedding(N_PIECE_IDS, d_model)
        self.square_emb = nn.Embedding(64, d_model)
        if use_aux:
            self.aux_proj = nn.Linear(AUX_DIM, d_model)

        self.state_proj = nn.Linear(STATE_DIM, d_model)
        self.skill_proj = nn.Linear(META_DIM, d_model)

        if use_history:
            self.hist_from_emb = nn.Embedding(65, d_model)
            self.hist_to_emb   = nn.Embedding(65, d_model)
            self.hist_age_emb  = nn.Embedding(k_history, d_model)

        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_head, n_buckets, dropout, use_gab) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.promo_head = nn.Linear(d_model, N_PROMO)
        self.d_head_scale = 1.0 / math.sqrt(d_model)

        self.register_buffer("move_ft", torch.from_numpy(MOVE_FT))
        self.register_buffer("move_promo_ft", torch.from_numpy(MOVE_PROMO_FT))
        self.register_buffer("is_promo", torch.from_numpy(IS_PROMO))

    def forward(self, piece_ids, state, meta, hist_from=None, hist_to=None, aux=None):
        B = piece_ids.shape[0]

        sq = self.piece_emb(piece_ids) + self.square_emb.weight.unsqueeze(0)
        if self.use_aux and aux is not None:
            sq = sq + self.aux_proj(aux)

        tokens = [sq, self.state_proj(state).unsqueeze(1), self.skill_proj(meta).unsqueeze(1)]

        if self.use_history:
            age = self.hist_age_emb.weight.unsqueeze(0)
            tokens.append(self.hist_from_emb(hist_from) + self.hist_to_emb(hist_to) + age)

        x = self.drop(torch.cat(tokens, dim=1))
        for block in self.blocks:
            x = block(x, self.gab_index)
        x = self.ln_f(x)

        sq_out = x[:, :64]
        q = self.q_proj(sq_out)
        k = self.k_proj(sq_out)
        from_to = (q @ k.transpose(1, 2) * self.d_head_scale).reshape(B, 64 * 64)
        promo = self.promo_head(sq_out).reshape(B, 64 * N_PROMO)

        logits = from_to[:, self.move_ft]
        logits = logits + torch.where(
            self.is_promo.unsqueeze(0),
            promo[:, self.move_promo_ft],
            torch.zeros((), device=logits.device, dtype=logits.dtype),
        )
        logits[:, :N_SPECIAL] = float("-inf")
        return logits

### Load the pretrained weights`strict=True` on purpose. A v1/v2/v3 checkpoint has entirely different keys, and loadingone non-strictly leaves a randomly initialised model that trains from scratch whilelooking like a fine-tune.

In [ ]:
model = Chessformer().to(device)
state_dict = torch.load(BASE_CKPT, map_location=device)
model.load_state_dict(state_dict, strict=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {BASE_CKPT}: {n_params / 1e6:.2f}M params, {model.n_tokens} tokens/position")

# Pretraining put ~1.2M positions against these weights; this run has a fraction of
# that, which is why the LR is low and the schedule is short.
print(f"Fine-tuning data: {tr_pos:,} positions vs {n_params:,} parameters "
      f"({tr_pos / n_params:.3f} positions/parameter)")

## 8. Baseline — the pretrained model on the held-out Carlsen games**Run this before training.** It is the only number the fine-tuned result can becompared against. v4's Lichess Elite Top-1 measures a different task on a differentdistribution; expect this baseline to come out *higher* than it, because elite play ismore predictable than the average 2200-2600 game.

In [ ]:
@torch.no_grad()
def evaluate_masked_topk(model, records, carlsen_only=True, k_values=(1, 3),
                         max_games_eval=None, desc="Evaluating"):
    """Legal-move-masked Top-k, one forward pass per game.

    Legal moves come from python-chess replaying the real game, never from the model's
    belief about the board. `unmasked_argmax_legal_rate` is the board-tracking
    diagnostic: how often the raw argmax over all 4546 moves is even legal.
    """
    model.eval()
    hits = {k: 0 for k in k_values}
    total = 0
    unmasked_legal_argmax = 0

    subset = records if max_games_eval is None else records[:max_games_eval]

    for rec in tqdm(subset, desc=desc, leave=False):
        data, legal_per_pos = extract_game_positions(rec, with_legal=True)
        target = data.pop("target")
        batch = {k: torch.from_numpy(v).to(device) for k, v in data.items()}

        probs = F.softmax(model(**batch).float(), dim=-1).cpu().numpy()
        mask = carlsen_mask(rec) if carlsen_only else np.ones(len(target), dtype=bool)

        for t in range(len(target)):
            if not mask[t]:
                continue
            legal_ids = legal_per_pos[t]
            if not legal_ids:
                continue
            p = probs[t]

            if int(p.argmax()) in set(legal_ids):
                unmasked_legal_argmax += 1

            ranked = [legal_ids[i] for i in np.argsort(-p[legal_ids])]
            for k in k_values:
                if target[t] in ranked[:k]:
                    hits[k] += 1
            total += 1

    results = {f"top_{k}": hits[k] / max(1, total) for k in k_values}
    results["unmasked_argmax_legal_rate"] = unmasked_legal_argmax / max(1, total)
    results["positions_evaluated"] = total
    return results


def report(name, res):
    print(f"{name}:")
    for k, v in res.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


t0 = time.time()
base_test = evaluate_masked_topk(model, test_records, CARLSEN_ONLY,
                                 max_games_eval=EVAL_MAX_GAMES, desc="baseline")
report(f"Pretrained v4, held-out Carlsen games "
       f"({'all moves' if TRAIN_BOTH_SIDES else 'his moves only'})", base_test)
print(f"({time.time() - t0:.0f}s)")

## 9. Fine-tuneSame loop as v4's Section 9 — per-batch scheduler stepping, gradient clipping,best-val-loss checkpoint restored at the end — with a lower LR, a shorter schedule and atighter early-stop patience. The per-epoch Top-1 probe is on a subset of the held-outgames, so you can watch the metric you actually care about while the loss moves.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps  = N_EPOCHS * len(train_loader)
WARMUP_STEPS = min(100, max(1, total_steps // 20))


def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(loader, train=True, step_scheduler=False):
    model.train(train)
    total_loss, total_n = 0.0, 0

    for batch in tqdm(loader, leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        target = batch.pop("target")
        if target.numel() == 0:
            continue

        with torch.set_grad_enabled(train):
            logits = model(**batch)
            loss = F.cross_entropy(logits, target)

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            if step_scheduler:
                scheduler.step()

        n = target.shape[0]
        total_loss += loss.item() * n
        total_n += n

    return total_loss / max(total_n, 1)


history = []

# Epoch 0 is the pretrained model, and it competes. Seeding best_val_loss with inf
# instead would let epoch 1 overwrite the base weights unconditionally -- so a run in
# which *every* epoch makes things worse would still save a damaged checkpoint, with
# no way back. Measured first, the base model wins ties and the worst case is a no-op.
base_val_loss = run_epoch(val_loader, train=False)
print(f"Epoch 0 (pretrained, no fine-tuning) | val ppl {math.exp(base_val_loss):.2f}"
      f"  <- fine-tuning has to beat this to be saved")

best_val_loss = base_val_loss
best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}
epochs_since_best = 0

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True, step_scheduler=True)
    val_loss   = run_epoch(val_loader, train=False)

    top1 = None
    if EVAL_GAMES_PER_EPOCH:
        top1 = evaluate_masked_topk(model, test_records, CARLSEN_ONLY,
                                    max_games_eval=EVAL_GAMES_PER_EPOCH,
                                    desc=f"epoch {epoch} top-1")["top_1"]
    history.append((epoch, train_loss, val_loss, top1))

    print(f"Epoch {epoch}/{N_EPOCHS} | train ppl {math.exp(train_loss):.2f} | "
          f"val ppl {math.exp(val_loss):.2f} | "
          f"ratio {math.exp(val_loss - train_loss):.2f} | "
          f"lr {scheduler.get_last_lr()[0]:.2e}"
          + (f" | test top-1 {top1:.4f}" if top1 is not None else "")
          + f" | {time.time() - t0:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}
        epochs_since_best = 0
    else:
        epochs_since_best += 1
        if epochs_since_best >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stop: no val improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

model.load_state_dict(best_state_dict)
print(f"\nRestored best checkpoint (val loss {best_val_loss:.4f}, "
      f"ppl {math.exp(best_val_loss):.2f})")
if best_val_loss >= base_val_loss:
    print("This is the PRETRAINED model -- no epoch beat it. Fine-tuning did not help "
          "on this data; see section 12 before retraining.")

In [ ]:
try:
    import matplotlib.pyplot as plt

    eps  = [h[0] for h in history]
    trl  = [h[1] for h in history]
    val  = [h[2] for h in history]
    top1 = [h[3] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(eps, trl, label="Train loss")
    axes[0].plot(eps, val, label="Val loss")
    axes[0].axvline(eps[int(np.argmin(val))], color="green", linestyle=":", label="Best checkpoint")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy")
    axes[0].set_title("Fine-tuning curve - Carlsen"); axes[0].legend(); axes[0].grid(alpha=.3)

    if any(t is not None for t in top1):
        axes[1].plot(eps, top1, marker="o", label="Held-out Top-1")
        axes[1].axhline(base_test["top_1"], color="red", linestyle="--",
                        label=f"Pretrained v4 ({base_test['top_1']:.3f})")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Top-1")
        axes[1].set_title("Does fine-tuning beat the base model?")
        axes[1].legend(); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not available; skipping the plot")

## 10. Final evaluationThree numbers matter:- **base → fine-tuned on held-out games** — did specialising help at all?- **train − test Top-1** — the overfit gap. ~170k positions against 6.5M parameters is  thin. A large gap means fewer epochs or a lower LR, *not* more training.- **`unmasked_argmax_legal_rate`** — should stay very high. If it drops, the fine-tune  is damaging the model's board tracking, which is the clearest sign the LR is too high.

In [ ]:
ft_test = evaluate_masked_topk(model, test_records, CARLSEN_ONLY,
                               max_games_eval=EVAL_MAX_GAMES, desc="finetuned/test")
report("Fine-tuned, held-out Carlsen games", ft_test)

ft_train = evaluate_masked_topk(model, train_records, CARLSEN_ONLY,
                                max_games_eval=EVAL_MAX_GAMES, desc="finetuned/train")
report("\nFine-tuned, train games (same sample size, for the overfit gap)", ft_train)

print("\n" + "=" * 72)
print("SUMMARY  -- all on the SAME held-out Carlsen positions")
print(f"  Top-1   base {base_test['top_1']:.4f}  ->  fine-tuned {ft_test['top_1']:.4f}"
      f"   ({ft_test['top_1'] - base_test['top_1']:+.4f})")
print(f"  Top-3   base {base_test['top_3']:.4f}  ->  fine-tuned {ft_test['top_3']:.4f}"
      f"   ({ft_test['top_3'] - base_test['top_3']:+.4f})")
print(f"  Legal-argmax  {base_test['unmasked_argmax_legal_rate']:.4f}  ->  "
      f"{ft_test['unmasked_argmax_legal_rate']:.4f}")
print(f"  Overfit gap (train - test Top-1): {ft_train['top_1'] - ft_test['top_1']:+.4f}")
print("\n  NOTE: not comparable to v4's Lichess Elite Top-1. Predicting one player is a")
print("        different task than predicting the average 2200-2600 human.")
print("=" * 72)

### Eyeball itA quick look at what the fine-tuned model wants to play from the start, and in a coupleof Carlsen-typical positions. Openings are where a player-specific model should differmost visibly from the base one.

In [ ]:
@torch.no_grad()
def top_moves(board, elo_w=MAX_ELO, elo_b=MAX_ELO, n=5):
    """Masked softmax over legal moves for an arbitrary board (no move history)."""
    T = 1
    piece_ids = np.zeros((T, 64), dtype=np.int64)
    for sq, pc in board.piece_map().items():
        piece_ids[0, sq] = PIECE_ID[(pc.color, pc.piece_type)]
    state = np.zeros((T, STATE_DIM), dtype=np.float32)
    state[0, 0] = 1.0 if board.turn == chess.WHITE else 0.0
    state[0, 1] = float(board.has_kingside_castling_rights(chess.WHITE))
    state[0, 2] = float(board.has_queenside_castling_rights(chess.WHITE))
    state[0, 3] = float(board.has_kingside_castling_rights(chess.BLACK))
    state[0, 4] = float(board.has_queenside_castling_rights(chess.BLACK))
    if board.ep_square is not None:
        state[0, 5 + chess.square_file(board.ep_square)] = 1.0

    meta = np.zeros((T, META_DIM), dtype=np.float32)
    e_w = min(1.0, max(0.0, (elo_w - MIN_ELO) / ELO_SPAN))
    e_b = min(1.0, max(0.0, (elo_b - MIN_ELO) / ELO_SPAN))
    wtm = board.turn == chess.WHITE
    meta[0, 0], meta[0, 1] = (e_w, e_b) if wtm else (e_b, e_w)
    meta[0, 5] = min(1.0, math.log1p(TC_BASE) / _TC_LOG_MAX)

    hist_from = np.full((T, K_HISTORY), NO_SQUARE, dtype=np.int64)
    hist_to   = np.full((T, K_HISTORY), NO_SQUARE, dtype=np.int64)
    stack = board.move_stack
    for k in range(K_HISTORY):
        j = len(stack) - 1 - k
        if j < 0:
            break
        hist_from[0, k], hist_to[0, k] = stack[j].from_square, stack[j].to_square

    feats = {"piece_ids": piece_ids, "state": state, "meta": meta,
             "hist_from": hist_from, "hist_to": hist_to}
    batch = {k: torch.from_numpy(v).to(device) for k, v in feats.items()}

    model.eval()
    logits = model(**batch)[0].float().cpu().numpy()
    legal = list(board.legal_moves)
    ids = np.array([stoi[m.uci()] for m in legal])
    p = np.exp(logits[ids] - logits[ids].max())
    p /= p.sum()
    order = np.argsort(-p)[:n]
    return [(board.san(legal[i]), float(p[i])) for i in order]


for line in ([], ["e2e4"], ["d2d4", "g8f6"]):
    b = chess.Board()
    for u in line:
        b.push(chess.Move.from_uci(u))
    label = " ".join(line) or "start"
    print(f"{label:<16} " + "  ".join(f"{s} {p:.3f}" for s, p in top_moves(b)))

## 11. Save and downloadWrites `chess_gpt_v4_carlsen.pt` and `chess_move_vocab_v4_carlsen.json`. The vocab fileis the v4 one plus fine-tuning metadata — same `itos`, same architecture fields — so`app_transformer.py` loads it with no code change.To serve it locally after downloading, from the repo root:```set CHESS_MODEL_PATH=chess_gpt_v4_carlsen.ptset CHESS_VOCAB_PATH=chess_move_vocab_v4_carlsen.jsonpython app_transformer.py```(`export` instead of `set` on Mac/Linux.) Leave the variables unset to go back to plain v4.

In [ ]:
torch.save(model.state_dict(), OUT_CKPT)

cfg = dict(vocab_data)          # same itos + architecture fields as v4
cfg.update({
    "finetuned_from": BASE_CKPT,
    "finetuned_on": os.path.basename(PGN_PATH),
    "finetune_player": "Carlsen,M",
    "finetune_scope": "all_moves" if TRAIN_BOTH_SIDES else "carlsen_moves_only",
    "finetune_games": len(train_records),
    "finetune_positions": tr_pos,
    "finetune_lr": LR,
    "finetune_epochs_run": len(history),
    "finetune_best_val_loss": best_val_loss,
    "finetune_top1_base": base_test["top_1"],
    "finetune_top1_test": ft_test["top_1"],
    "finetune_top3_test": ft_test["top_3"],
    "finetune_tc_base": TC_BASE,
})
with open(OUT_VOCAB, "w") as f:
    json.dump(cfg, f)

print(f"Saved {OUT_CKPT}   ({os.path.getsize(OUT_CKPT) / 1e6:.1f} MB)")
print(f"Saved {OUT_VOCAB}")

In [ ]:
# Download both. On Colab the runtime is wiped when it disconnects -- do this now.
if IN_COLAB:
    from google.colab import files
    files.download(OUT_CKPT)
    files.download(OUT_VOCAB)
else:
    print(f"Not on Colab; the files are already in {os.getcwd()}")

# Alternative: copy to Google Drive instead of downloading.
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy(OUT_CKPT,  "/content/drive/MyDrive/")
# shutil.copy(OUT_VOCAB, "/content/drive/MyDrive/")

## 12. What to try if the delta is smallIn rough order of expected payoff:1. **`CHRONOLOGICAL_SPLIT = True`.** A random split lets the model see 2021 Carlsen while   testing on 2021 Carlsen. Holding out his most recent years is the harder, more honest   question, and the gap between the two splits tells you how much of the gain is style   versus memorised openings.2. **Fewer epochs / lower LR.** With ~170k positions the overfit gap in Section 10 usually   binds before the model has finished learning anything new. Check it before adding work.3. **`TRAIN_BOTH_SIDES = True`** as a control. If it scores the same on Carlsen's moves,   the model is not learning anything player-specific and the win is just domain shift   (elite classical chess vs Lichess blitz).4. **More games.** 4,300 games is small. Adding other elite players with a per-player   token would let the pretrained model share structure across them instead of   specialising 6.5M parameters on one person — the Maia-2 approach.5. **Freeze the lower blocks.** Training only `ln_f`, `q_proj`, `k_proj`, `promo_head` and   the last block or two keeps the board understanding intact and moves only the policy.   Worth trying if `unmasked_argmax_legal_rate` fell in Section 10.The ELO conditioning is effectively dead on this data (everything clamps to the top of theband), so the rating slider in the web app will do very little with this checkpoint. If youwant it back, the fix is more players at more ratings, not a wider band.